<a href="https://colab.research.google.com/github/spd-creates/melanies_smoothies/blob/main/RAG_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#install dependencies
!pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


Chat Model

In [ ]:
pip install -qU 'langchain[openai]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 8.7 MB/s eta 0:00:00


In [ ]:
pip install -qU langchain-google-genai

In [ ]:
import getpass
import os

# Set API key if not already present
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Enter API key for Gemini: ')

from langchain.chat_models import init_chat_model

# Changes: Updated model_provider to 'google_genai' and model name to 'gemini-3-flash-preview'
llm = init_chat_model(
    'gemini-3-flash-preview',
    model_provider='google_genai'
)

Enter API key for Gemini: ··········


In [ ]:
import getpass
import os

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter API key for OpenAI: ')

from langchain.chat_models import init_chat_model

llm = init_chat_model('gpt-4o-mini', model_provider='openai')

Enter API key for OpenAI: ··········


Embedding Model

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model='text-embedding-3-large')

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Use Google's embedding model (e.g., 'text-embedding-004' or 'gemini-embedding-2-preview')
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2"
)

Vector database

In [ ]:
pip install -qU langchain-community

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS(embedding_function=embeddings)

TypeError: FAISS.__init__() missing 3 required positional arguments: 'index', 'docstore', and 'index_to_docstore_id'

RAG Q/A over Web Content

In [ ]:
!pip install -U langchainhub

In [ ]:
pip install -qU faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 39.1 MB/s eta 0:00:00


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
import bs4
# from langchain import hub
import langchainhub as hub
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.prompts import ChatPromptTemplate

# Load and chunk contents of the blog
loader = WebBaseLoader(
    #  web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/"),
     web_paths=["https://lilianweng.github.io/posts/2023-06-23-agent/"],
     bs_kwargs=dict(
         parse_only=bs4.SoupStrainer(
            # class_("post-content", "post-title", "post-header")
          class_=["post-content", "post-title", "post-header"]
         )
     ),
 )
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

# Index chunks
# _ = vector_store.add_documents(documents=all_splits)
vector_store = FAISS.from_documents(documents=all_splits, embedding=embeddings)

# Define prompt for question-answering
# prompt = hub.pull("rlm/rag-prompt")

# 3. MANUAL PROMPT (Replaces the broken hub.pull)
# This is exactly what "rlm/rag-prompt" contains:
template = """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# Define state for application
class State(TypedDict):
     question: str
     context: List[Document]
     answer: str

# Define application steps
def retrieve(state: State):
     retrieved_docs = vector_store.similarity_search(state["question"])
     return {"context": retrieved_docs}

def generate(state: State):
     docs_content = "\n\n".join(doc.page_content for doc in state["context"])
     messages = prompt.invoke({"question": state["question"], "context": docs_content})
     response = llm.invoke(messages)
     return {"answer": response.content}

# Compile application and test
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

response = graph.invoke({"question": "What is Task Decomposition?"})
print(response["answer"])

[{'type': 'text', 'text': 'Task decomposition is the process of breaking down complex problems into smaller, more manageable steps or subgoals to simplify execution. It can be achieved through simple prompting like "thinking step by step," task-specific instructions, human inputs, or techniques like Chain of Thought (CoT) and Tree of Thoughts. Additionally, it can involve parsing user requests into structured tasks with specific attributes or outsourcing planning to external tools using PDDL.', 'extras': {'signature': 'EpEQCo4QAQw51se4Bl+MZTBvyhPzV11BH84KmM+obmFiRWcl4LVWvxOpXwdnvW9jtospG/T7NLVIpXBJUZ7RZuicH/gXwD8sShvzcEj9Y5JjruVdYGFn9RqG/iZOPlSVFxdSAlMbJqWto4Wag+4zaCX1Yxil8MrMvZG797xy5OUJhBUSe/JIXu1nkXWPhCgMLkqYEb4aibhWpxqEt688AFq9LBhdkb7QQ0O5YQ5NtTiBZaMAtgCFgIQ5NvRMuztmkIjY8v2yPy59cg4oJxlrhO7M+xCIdFsENw3kfjPgztZztA9pHfbVRRCpNFoHw3kSOysCTAmScnaj5g2caorHI0ETydJHms1XCsXrn7WObOTGLtES2XdQB9xshiY3M40djhvjSpdsyLi/o9GfwyV5FyDjMI7sTT0xG2OC5TK0qsrGDD4bJYiDhnUXDF/DIAifkxtzB7D68U33y9IsfMynD3gJQa8

Models:

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2